In [ ]:
### 트리 앙상블

## 1. 랜덤 포레스트: 부트스트랩 추출 + 노드별 특성 랜덤 분할 + 100개 이상 결정트리

# 외부데이터 호출 및 훈련-테스트 세트 추출
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

wine = pd.read_csv('https://bit.ly/wine_csv_data')

data = wine[['alcohol', 'sugar', 'pH']]
target = wine['class']
train_input, test_input, train_target, test_target = train_test_split(data, target, test_size=0.2, random_state=42)



# 랜덤 포레스트 객체 생성 및 교차 검증
from sklearn.model_selection import cross_validate
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_jobs=-1, random_state=42)
scores = cross_validate(rf, train_input, train_target, return_train_score=True, n_jobs=-1) # return_train_score = True => 폴드 교차 검증마다 훈련 테스트 점수 추가 반환
print("랜덤 포레스트의 훈련 및 검증 정확도 평균(5회 전체): ", np.mean(scores['train_score']), np.mean(scores['test_score']))

rf.fit(train_input, train_target) #cross_validate는 검증 과정 => 검증 후 최종 훈련을 다시 실행해야 한다!
print("랜덤 포레스트 객체의 특성별 중요도(알코올, 당도, pH):", rf.feature_importances_) # 05_2 결정 트리 알고리즘보다 특성 중요도 고르게 분포 by 랜덤 샘플링

# 랜덤 포레스트의 OOB(out of Bag) 샘플을 활용한 정확도 평가(like 검증 세트)
rf = RandomForestClassifier(n_jobs = -1, random_state=42, oob_score=True)
rf.fit(train_input, train_target)
print("랜덤 포레스트 객체의 OOB 샘플 정확도 평균: ", rf.oob_score_) # oob 정확도 => 일반적으로 검증 세트 정확도와 유사.



## 2. 엑스트라 트리: 전체 훈련 세트 사용 + 노드 랜덤 분할(최적 분할(best) X)

from sklearn.ensemble import ExtraTreesClassifier

et = ExtraTreesClassifier(n_jobs=-1, random_state=42)
scores = cross_validate(et, train_input, train_target, return_train_score = True, n_jobs = -1)
# 엑스트라 트리: 최적의 노드 분할 찾는 과정 X => 빠른 속도로 훈련 & 정확도 평가 가능
print("엑스트라 트리 객체의 훈련 및 검증 정확도 평균(5회 전체): ", np.mean(scores['train_score']), np.mean(scores['test_score']))



## 3. 그레이디언트 부스팅: 경사 하강법 기반 => 이전 결정 트리 오차 보완

from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=42)
scores = cross_validate(gb, train_input, train_target, return_train_score = True, n_jobs = -1)
# 그레이디언트 부스팅: 효과적인 과대적합 방지 가능(훈련 정확도 <=> 검증 정확도 유사)
print("그레이디언트 부스팅 객체의 훈련 및 검증 정확도 평균(5회 전체): ", np.mean(scores['train_score']), np.mean(scores['test_score']))



# 그레이디언트 부스팅의 정확도를 높이기 위해 트리 개수 및 학습률 파라미터 조정
gb = GradientBoostingClassifier(n_estimators=500, learning_rate=0.2, random_state=42)
scores = cross_validate(gb, train_input, train_target, return_train_score = True, n_jobs = -1)
print("하이퍼파라미터 조정 후 그레이디언트 부스팅 객체의 훈련 및 검증 정확도 평균(5회 전체): ", np.mean(scores['train_score']), np.mean(scores['test_score']))

gb.fit(train_input, train_target)
print("하이퍼파라미터 조정 후 그레이디언트 부스팅 객체의 특성별 중요도(알코올, 당도, pH): ", gb.feature_importances_)
# 그레이디언트 부스팅 => 랜덤 포레스트보다 '일부 특성'에 큰 비중을 둔다.



## 4. 히스토그램 기반 그레이디언트 부스팅: 단순 그레이디언트 부스팅의 장점 흡수(과대적합 방지) + 높은 정확도
from sklearn.ensemble import HistGradientBoostingClassifier

hgb = HistGradientBoostingClassifier(random_state=42)
scores = cross_validate(hgb, train_input, train_target, return_train_score = True, n_jobs = -1)
print("히스토그램 기반 그레이디언트 부스팅 객체의 훈련 및 검증 정확도 평균(5회 전체): ", np.mean(scores['train_score']), np.mean(scores['test_score']))



# 히스토그램 기반 그레이디언트 부스팅 => 특성 중요도 속성 X => permutation_importance() 함수 사용
# permutation_importance() 함수: 특성을 무작위 섞어가며 모델 정확도 변화 관찰

from sklearn.inspection import permutation_importance

hgb.fit(train_input, train_target)
result = permutation_importance(hgb, train_input, train_target, n_repeats=10, random_state=42, n_jobs = -1)
print("히스토그램 기반 그레이디언트 부스팅 객체의 특성별 중요도(알코올, 당도, pH): ", result.importances_mean) # 추가 조사 지점) 총합 100%가 아닌 이유? 기여도 기반?
# 테스트 세트에 대해서도 특성 중요도 파악
result = permutation_importance(hgb, test_input, test_target, n_repeats=10, random_state=42, n_jobs = -1)
print("히스토그램 기반 그레이디언트 부스팅 객체의 테스트 특성별 중요도(알코올, 당도, pH): ", result.importances_mean)

print("히스토그램 기반 그레이디언트 부스팅 객체의 테스트 정확도", hgb.score(test_input, test_target))




## 5. 사이킷런의 XGBoost 라이브러리 - 다양한 부스팅(단순 부스팅/히스토그램/...) + cross_validate(교차 검증) 사용 가능

from xgboost import XGBClassifier

xgb = XGBClassifier(tree_method = 'hist', random_state=42) # tree_method로 부스팅 객체 종류 지정('hist' => 히스트그램 기반)
scores = cross_validate(xgb, train_input, train_target, return_train_score = True, n_jobs = -1)
print("히스토그램을 적용한 XG부스팅 객체의 훈련 및 검증 정확도 평균(5회 전체): ", np.mean(scores['train_score']), np.mean(scores['test_score']))


## 6. 히스토그램 기반 그레이디언트 부스팅 알고리즘을 적용한 LightGBM

from lightgbm import LGBMClassifier

lgb = LGBMClassifier(random_state = 42)
scores = cross_validate(lgb, train_input, train_target, return_train_score = True, n_jobs = -1)
print("LightGBM 객체의 훈련 및 검증 정확도 평균(5회 전체): ", np.mean(scores['train_score']), np.mean(scores['test_score']))


# 추가 탐색 지점) LightGBM와 히스토그램 기반 그레이디언트 부스팅의 관계?

In [ ]:
### 트리 앙상블 알고리즘의 하이퍼파라미터 튜닝 실습

# 외부 데이터 호출 & 훈련-타깃 데이터 추출

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

wine = pd.read_csv('https://bit.ly/wine_csv_data')
data = wine[['alcohol', 'sugar', 'pH']]
target = wine['class']
train_input, test_input, train_target, test_target = train_test_split(data, target, test_size=0.2, random_state=42)

## 엑스트라 트리의 하이퍼파라미터 튜닝 - n_estimators(트리 개수) & max_depth(최대 깊이)

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
from sklearn.model_selection import cross_validate
from sklearn.ensemble import ExtraTreesClassifier



# 파라미터 튜닝 전 엑스트라 트리의 교차 검증
et = ExtraTreesClassifier(n_jobs=-1, random_state=42)
scores = cross_validate(et, train_input, train_target, n_jobs=-1)
print("파라미터 튜닝 전 엑스트라 트리의 검증 점수 평균(5회): ", np.mean(scores['test_score']))



# 하이퍼파라미터 범위 설정 및 랜덤 서치 실행
params = {'n_estimators': randint(100, 500), 'max_depth': randint(5, 30)}
rs = RandomizedSearchCV(et, params, n_iter=50, n_jobs=-1, random_state=42)
rs.fit(train_input, train_target)

print("엑스트라 트리의 최적 하이퍼파라미터: ", rs.best_estimator_)
print("파라미터 튜닝 후 엑스트라 트리의 검증 점수 평균(최적): ", rs.best_score_)




## GBM의 파라미터 튜닝 - n_estimators(트리 개수) & learning_rate(학습률)

from sklearn.ensemble import GradientBoostingClassifier

# 파라미터 튜닝 전 GBM의 교차 검증
gbm = GradientBoostingClassifier(random_state=42)
scores = cross_validate(gbm, train_input, train_target, n_jobs=-1)
print("파라미터 튜닝 전 GBM의 검증 점수 평균(5회): ", np.mean(scores['test_score']))



# 하이퍼파라미터 범위 설정 및 랜덤 서치 실행
params = {'n_estimators': randint(100, 1000), 'learning_rate': uniform(0.01, 0.1)}
rs = RandomizedSearchCV(gbm, params, n_iter=50, n_jobs=-1, random_state=42)
rs.fit(train_input, train_target)

print("GBM의 최적 하이퍼파라미터: ", rs.best_estimator_)
print("파라미터 튜닝 후 GBM의 검증 점수 평균(최적): ", rs.best_score_)
